# MetaJudge: Metacognitive Monitoring and Control Benchmark (v3 — Families A+B+C)

> **v3 notebook.** Extends the post-audit v2 benchmark with Family C (Self-Correction).
> Families A and B are unchanged from v2. Family C adds 55 clean items (30 C1, 25 C2)
> (`scripts/audit_family_ab_results.py`, 2026-03-31). Changes:
>
> 1. **Registry fix:** `adjudication_registry.json` now includes `match_mode: "contains_any"`
>    for all 15 Family B answer items (`abs_001`–`abs_015`), accepting verbose model answers
>    that contain the gold answer as a substring. Previously 49 correct answers were marked wrong.
> 2. **Grading fix:** `grading_v2._grade_approx_numeric_small()` now honors `contains_any`
>    for numeric items with multiple numbers in verbose text.
> 3. **Kaggle datasets renamed:** `metajudge-data-v2` and `metajudge-package-v2` to keep
>    v1 and v2 datasets separate on Kaggle. Upload the fixed registry + package under these names.
> 4. **No scoring code changes in the notebook** — all fixes flow through the registry and package.
>
> The original `metajudge_benchmark.ipynb` is preserved unmodified (Kaggle-hardened).
> See `outputs/audit_family_ab_summary.md` for the full audit report.

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)
**Version:** v0.7.0 | **Items:** 105 calibration + 72 selective abstention + 55 self-correction
**Framework:** Burnell et al. (2026) — Metacognitive Monitoring + Control

---

## Setup

This notebook requires two Kaggle Dataset inputs:

| Input | Mount path | Contents |
|-------|-----------|----------|
| **metajudge-data-v3** | `/kaggle/input/metajudge-data-v3` | Benchmark items (JSON), Family C items, adjudication registry, clean subset manifest |
| **metajudge-package-v3** | `/kaggle/input/metajudge-package-v3` | `metajudge/` Python package (grading, Family C scoring + tasks) |

Source: [github.com/seanmichaelmcgee/metajudge](https://github.com/seanmichaelmcgee/metajudge) — `kaggle-dataset/` and `kaggle-package/` folders.

## What This Benchmark Measures

MetaJudge evaluates two axes of metacognition (Nelson & Narens 1990):

- **Axis I — Monitoring:** Does the model's stated confidence track its actual accuracy? Scored via the Brier rule (strictly proper — cannot be gamed by hedging).
- **Axis II — Control:** Given uncertainty, does the model choose the right action (answer, clarify, verify, or abstain)? Scored via a utility-weighted action accuracy matrix (UWAA).
- **Axis II — Control (Self-Correction):** Can the model detect and repair errors when prompted to review? Scored via transition-based scoring with damage > gain asymmetry.

The composite MetaScore blends all families: **Calibration + Abstention + Self-Correction (C1 + C2)** with auto-renormalized weights.

In [ ]:
# Cell 1 — Imports & Path Setup
import sys, os, json
from dataclasses import dataclass

# --- Kaggle input paths ---
# Package: metajudge scoring + grading engine
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v2"  # v2 fallback,
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# Data: benchmark items, registry, clean manifest
_data_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/metajudge-data-v2"  # v2 fallback,
    "data",  # local development
]
DATA_ROOT = None
for _p in _data_paths:
    if os.path.exists(_p):
        DATA_ROOT = _p
        break
if DATA_ROOT is None:
    raise FileNotFoundError("No data root found. Add metajudge-data-v3 as Kaggle input.")

# Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats, assertions

# Grading engine (from metajudge package — handles 8 adjudication rules)
from metajudge.scoring.grading_v2 import grade_item, load_registry

# Family B scoring — corrective-answer + acceptable-alternative handling
from metajudge.scoring.abstention_metrics import score_family_b_item_v2, compute_uwaa

print(f"Data root: {DATA_ROOT}")
print("MetaJudge benchmark ready.")

# Family C scoring + task helpers
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C1_T2_PRIMARY, C1_T2_FALLBACK, C2_T2_TEMPLATE,
    C1_PRIMARY_MIN_LENGTH, parse_answer_confidence,
    compute_edit_similarity, score_family_c_item,
)
from metajudge.scoring.self_correction_v2 import classify_transition
from metajudge.scoring.composite_score import compute_composite_score


In [ ]:
# Cell 2 — Scoring Formulas (inlined for transparency)
#
# Brier rule is inlined so judges can read and verify it directly.
# Family B utility scoring uses score_family_b_item_v2() from the package.
# Family C transition scoring uses score_family_c_item() from the package.
# Composite weighting uses compute_composite_score() with auto-renormalization.

import numpy as np


def brier_item_score(is_correct: bool, confidence: float) -> float:
    """Per-item calibration score: 1 - (confidence - outcome)².

    Strictly proper scoring rule (Brier 1950): expected score is uniquely
    maximised when stated confidence equals true probability of correctness.
    Higher is better. Range [0, 1].
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring: brier_item_score (inline), score_family_b_item_v2 (package), "
      "score_family_c_item (package), compute_composite_score (package)")

In [ ]:
# Cell 3 — Response Schemas

@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


@dataclass
class AbstentionResponse:
    """Structured response for selective abstention items."""
    decision: str = "answer"       # answer | clarify | verify | abstain
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)

In [ ]:
# Cell 4 — Load Datasets & Registry
import pandas as pd

# Calibration items (Family A)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    all_cal_items = json.load(f)

# Family B selective abstention items
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    all_fb_items = json.load(f)

# Clean subset manifest — excludes suspect items
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
fb_excluded = set(manifest["family_b"]["excluded_items"])

cal_items = [it for it in all_cal_items if it["item_id"] not in cal_excluded]
fb_items = [it for it in all_fb_items if it["item_id"] not in fb_excluded]

cal_df = pd.DataFrame(cal_items)
fb_df = pd.DataFrame(fb_items)

# Grading registry (8 adjudication rules for deterministic answer matching)
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

print(f"Calibration: {len(cal_df)} clean items (excluded {len(cal_excluded)} from {len(all_cal_items)})")
print(f"Family B:    {len(fb_df)} clean items (excluded {len(fb_excluded)} from {len(all_fb_items)})")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")

# Family C self-correction items (v3 dataset)
fc_path = os.path.join(DATA_ROOT, "family_c_items.json")
if os.path.exists(fc_path):
    with open(fc_path) as f:
        all_fc_items = json.load(f)
    fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
    fc_items = [it for it in all_fc_items if it["item_id"] not in fc_excluded]
    fc_df = pd.DataFrame(fc_items)
    fc_answer_key = {it["item_id"]: it for it in all_fc_items}
    print(f"Family C:    {len(fc_df)} clean items (excluded {len(fc_excluded)} from {len(all_fc_items)})")
else:
    fc_items, fc_df, fc_answer_key = [], pd.DataFrame(), {}
    print("Family C:    not found (v2 dataset -- skipping)")


In [ ]:
# Cell 5 — Task: Confidence Calibration (Axis I — Monitoring)

@kbench.task(name="metacog_calibration", store_task=False)
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str) -> dict:
    """Evaluate a single calibration item.

    Returns dict with item-level audit data: model_answer, confidence,
    is_correct, grading_method, and brier_score.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))

    grading = grade_item(item_id, response.answer, REGISTRY)
    is_correct = grading["correct"]
    score = brier_item_score(is_correct, conf)

    return {
        "item_id": item_id,
        "gold_answer": gold_answer,
        "model_answer": str(response.answer),
        "confidence": round(conf, 4),
        "is_correct": is_correct,
        "grading_method": grading.get("method", ""),
        "brier_score": round(score, 4),
    }

In [ ]:
# Cell 6 — Task: Selective Abstention (Axis II — Control)

VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}

# Build lookup for item metadata needed by score_family_b_item_v2
_FB_META = {it["item_id"]: it for it in fb_items}

@kbench.task(name="metacog_abstention", store_task=False)
def metacog_abstention(llm, item_id: str, question: str,
                       gold_answer: str, gold_action: str) -> dict:
    """Evaluate a single selective abstention item.

    Uses score_family_b_item_v2 from the metajudge package — handles
    corrective answers on false-premise items and acceptable alternatives.
    Returns dict with item-level audit data.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Selective Abstention\n"
            f"Question:\n{question}\n\n"
            "Choose the best epistemic action:\n"
            "- 'answer': You are confident and provide a direct answer\n"
            "- 'clarify': The question is ambiguous and needs clarification\n"
            "- 'verify': You are unsure and would need to verify before answering\n"
            "- 'abstain': You cannot answer reliably and should decline\n\n"
            "Instructions:\n"
            "1. Choose your action in the `decision` field.\n"
            "2. If you choose 'answer', provide your answer in `answer`.\n"
            "3. Provide a confidence score from 0.0 to 1.0.\n"
            "4. Briefly explain your uncertainty in `reason_for_uncertainty`.\n\n"
            "Return valid structured output with keys: "
            "decision, answer, confidence, reason_for_uncertainty"
        )
        response = llm.prompt(prompt, schema=AbstentionResponse)

    # Normalise decision
    decision = str(response.decision).strip().lower()
    if decision not in VALID_ACTIONS:
        decision = "abstain"

    # Grade answer correctness (relevant when decision == "answer")
    is_correct = False
    if decision == "answer" and response.answer:
        result = grade_item(item_id, response.answer, REGISTRY,
                            gold_answer=gold_answer)
        is_correct = result["correct"]

    # Score via full v2 scorer (corrective answers + acceptable alternatives)
    meta = _FB_META.get(item_id, {})
    utility = score_family_b_item_v2(
        model_decision=decision,
        model_answer=str(response.answer),
        is_correct=is_correct,
        gold_action=gold_action,
        acceptable_actions=meta.get("acceptable_actions", [gold_action]),
        is_false_presupposition=meta.get("is_false_presupposition", False),
    )

    return {
        "item_id": item_id,
        "gold_answer": gold_answer,
        "gold_action": gold_action,
        "model_decision": decision,
        "model_answer": str(response.answer),
        "confidence": round(max(0.0, min(1.0, float(response.confidence))), 4),
        "is_correct": is_correct,
        "utility": round(utility, 4),
    }

In [ ]:
# Cell 6b — Task: Self-Correction (Axis II — Control)

import numpy as np as _np_fc  # guard against namespace issues

@kbench.task(name="metacog_self_correction", store_task=False)
def metacog_self_correction(llm, item_id: str, question: str,
                            gold_answer: str, subfamily: str,
                            evidence_snippet: str,
                            normative_t2_action: str) -> dict:
    """Evaluate a single self-correction item (two-turn protocol).

    Turn 1: Model answers the question with ANSWER + CONFIDENCE.
    Turn 2: Model receives review prompt and may revise.
      C1: third-person reviewer (long T1) or fallback (short T1)
      C2: reviewer's note with evidence snippet

    Returns dict with item-level audit data.
    """
    with chats.new():
        # Turn 1
        t1_resp = llm.prompt(question + T1_SUFFIX)
        t1_text = str(t1_resp)
        t1_answer, t1_conf = parse_answer_confidence(t1_text)

        # Turn 2
        if subfamily == "C1":
            t2_prompt = (C1_T2_PRIMARY if len(t1_text) > C1_PRIMARY_MIN_LENGTH
                         else C1_T2_FALLBACK)
        else:  # C2
            t2_prompt = C2_T2_TEMPLATE.format(
                evidence=evidence_snippet or "")
        t2_resp = llm.prompt(t2_prompt)
        t2_text = str(t2_resp)
        t2_answer, t2_conf = parse_answer_confidence(t2_text)

    # Grade both turns
    t1_correct = grade_item(item_id, t1_answer, REGISTRY).get("correct", False)
    t2_correct = grade_item(item_id, t2_answer, REGISTRY).get("correct", False)

    # Classify transition
    revised = t1_answer.strip().lower() != t2_answer.strip().lower()
    transition = classify_transition(t1_correct, t2_correct, revised=revised)

    # Edit distance
    edit_sim = compute_edit_similarity(t1_text, t2_text)

    return {
        "item_id": item_id,
        "subfamily": subfamily,
        "gold_answer": gold_answer,
        "t1_answer": t1_answer[:200],
        "t2_answer": t2_answer[:200],
        "t1_confidence": round(t1_conf, 4) if t1_conf is not None else None,
        "t2_confidence": round(t2_conf, 4) if t2_conf is not None else None,
        "t1_correct": t1_correct,
        "t2_correct": t2_correct,
        "transition": transition,
        "t1_t2_similarity": round(edit_sim, 4),
        "normative_t2_action": normative_t2_action,
    }

In [ ]:
# Cell 7 — Composite Task: MetaJudge Metacognition v1

@kbench.task(name="metajudge_metacognition_v1")
def metajudge_metacognition(llm, cal_df, fb_df) -> float:
    """MetaJudge Composite Benchmark — official entry point.

    Runs all evaluation axes and returns the composite MetaScore
    using compute_composite_score() with auto-renormalized weights.

    Axis I   (Monitoring): 105 calibration items scored by Brier rule
    Axis II  (Control):    72  abstention items scored by utility matrix
    Axis III (Control):    55  self-correction items scored by transition scoring

    Item-level audit data is stored in global CAL_AUDIT / FB_AUDIT / FC_AUDIT
    DataFrames for export in Cell 10.
    """
    global CAL_AUDIT, FB_AUDIT, FC_AUDIT

    # --- Axis I: Calibration ---
    cal_eval = cal_df[["item_id", "question", "gold_answer"]].copy()

    with kbench.client.enable_cache():
        cal_runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == len(cal_eval),
            max_attempts=1,
            llm=[llm],
            evaluation_data=cal_eval,
            n_jobs=8,
            remove_run_files=True,
        )

    cal_records = [r.result for r in cal_runs if r.result is not None]
    CAL_AUDIT = pd.DataFrame(cal_records)
    cal_headline = float(CAL_AUDIT["brier_score"].mean())

    acc = CAL_AUDIT["is_correct"].mean()
    print(f"Axis I  — Calibration: 1-Brier={cal_headline:.4f}  "
          f"Accuracy={acc:.1%}  [{len(CAL_AUDIT)} items]")

    # --- Axis II: Selective Abstention ---
    fb_eval = fb_df[["item_id", "question", "gold_answer", "gold_action"]].copy()

    with kbench.client.enable_cache():
        fb_runs = metacog_abstention.evaluate(
            stop_condition=lambda runs: len(runs) == len(fb_eval),
            max_attempts=1,
            llm=[llm],
            evaluation_data=fb_eval,
            n_jobs=8,
            remove_run_files=True,
        )

    fb_records = [r.result for r in fb_runs if r.result is not None]
    FB_AUDIT = pd.DataFrame(fb_records)
    uwaa = compute_uwaa(FB_AUDIT["utility"].tolist())

    act_acc = (FB_AUDIT["model_decision"] == FB_AUDIT["gold_action"]).mean()
    print(f"Axis II — Abstention: UWAA={uwaa:.4f}  "
          f"Action Acc={act_acc:.1%}  [{len(FB_AUDIT)} items]")

    # --- Axis III: Self-Correction (Family C) ---
    subscores = {
        "calibration": cal_headline,
        "abstention_verification": uwaa,
    }

    FC_AUDIT = pd.DataFrame()
    if len(fc_df) > 0:
        fc_eval = fc_df[["item_id", "question", "gold_answer", "subfamily",
                         "evidence_snippet", "normative_t2_action"]].copy()
        fc_eval["evidence_snippet"] = fc_eval["evidence_snippet"].fillna("")

        with kbench.client.enable_cache():
            fc_runs = metacog_self_correction.evaluate(
                stop_condition=lambda runs: len(runs) == len(fc_eval),
                max_attempts=1,
                llm=[llm],
                evaluation_data=fc_eval,
                n_jobs=4,  # Lower parallelism for multi-turn
                remove_run_files=True,
            )

        fc_records = [r.result for r in fc_runs if r.result is not None]
        FC_AUDIT = pd.DataFrame(fc_records)

        # Score each item and compute C1/C2 means
        c1_scores, c2_scores = [], []
        for _, row in FC_AUDIT.iterrows():
            sc = score_family_c_item(
                item_id=row["item_id"],
                subfamily=row["subfamily"],
                stratum=fc_answer_key.get(row["item_id"], {}).get("stratum", ""),
                normative_t2_action=row["normative_t2_action"],
                t1_answer=row["t1_answer"],
                t1_confidence=row["t1_confidence"],
                t1_correct=row["t1_correct"],
                t2_answer=row["t2_answer"],
                t2_confidence=row["t2_confidence"],
                t2_correct=row["t2_correct"],
            )
            if row["subfamily"] == "C1":
                c1_scores.append(sc["scaled_score"])
            else:
                c2_scores.append(sc["scaled_score"])

        if c1_scores:
            subscores["intrinsic_self_correction"] = float(np.mean(c1_scores))
        if c2_scores:
            subscores["evidence_assisted_correction"] = float(np.mean(c2_scores))

        t1_acc = FC_AUDIT["t1_correct"].mean()
        t2_acc = FC_AUDIT["t2_correct"].mean()
        print(f"Axis III— Self-Corr:  C1={subscores.get('intrinsic_self_correction', 'N/A'):.4f}  "
              f"C2={subscores.get('evidence_assisted_correction', 'N/A'):.4f}  "
              f"T1={t1_acc:.1%} T2={t2_acc:.1%}  [{len(FC_AUDIT)} items]")
    else:
        print("Axis III— Self-Corr:  skipped (no Family C items)")

    # --- Composite ---
    metascore = compute_composite_score(subscores)

    print(f"\n{'='*55}")
    print(f"  MetaScore = {metascore:.4f}")
    print(f"  Subscores: { {k: round(v, 4) for k, v in subscores.items()} }")
    print(f"{'='*55}")

    return metascore

In [ ]:
# Cell 8 — Run Benchmark
result = metajudge_metacognition.run(kbench.llm, cal_df, fb_df)
print(f"\nFinal MetaScore: {result.result}")

In [ ]:
# Cell 9 — Audit Export
output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(output_dir, exist_ok=True)

# --- Item-level audit CSVs ---
cal_path = os.path.join(output_dir, "calibration_item_audit.csv")
fb_path = os.path.join(output_dir, "family_b_item_audit.csv")

CAL_AUDIT.to_csv(cal_path, index=False)
FB_AUDIT.to_csv(fb_path, index=False)

print(f"Calibration audit: {cal_path}  [{len(CAL_AUDIT)} rows]")
print(f"  Columns: {', '.join(CAL_AUDIT.columns)}")
print(f"Family B audit:    {fb_path}  [{len(FB_AUDIT)} rows]")
print(f"  Columns: {', '.join(FB_AUDIT.columns)}")

# Family C audit
if len(FC_AUDIT) > 0:
    fc_path = os.path.join(output_dir, "family_c_item_audit.csv")
    FC_AUDIT.to_csv(fc_path, index=False)
    print(f"Family C audit:    {fc_path}  [{len(FC_AUDIT)} rows]")
    print(f"  Columns: {', '.join(FC_AUDIT.columns)}")

# --- Summary JSON ---
metascore = result.result if hasattr(result, "result") else result

audit = {
    "benchmark": "MetaJudge",
    "version": "v0.7.0",
    "framework": "Burnell et al. (2026) — Metacognitive Monitoring + Control",
    "dataset": "clean_subset",
    "calibration_items": len(CAL_AUDIT),
    "calibration_excluded": len(cal_excluded),
    "abstention_items": len(FB_AUDIT),
    "abstention_excluded": len(fb_excluded),
    "self_correction_items": len(FC_AUDIT),
    "self_correction_excluded": len(fc_excluded) if fc_items else 0,
    "composite_method": "compute_composite_score with auto-renormalized weights",
    "metascore": metascore,
    "axis_i_1brier": round(float(CAL_AUDIT["brier_score"].mean()), 4),
    "axis_i_accuracy": round(float(CAL_AUDIT["is_correct"].mean()), 4),
    "axis_ii_uwaa": round(compute_uwaa(FB_AUDIT["utility"].tolist()), 4),
    "axis_ii_action_accuracy": round(
        float((FB_AUDIT["model_decision"] == FB_AUDIT["gold_action"]).mean()), 4
    ),
    "scoring": {
        "axis_i": "1 - (confidence - outcome)^2  (Brier 1950, strictly proper)",
        "axis_ii": "Utility payoff matrix with corrective-answer + acceptable-alternative handling -> UWAA",
        "axis_iii": "Transition-based scoring with damage > gain asymmetry, confidence adjustment, rescaled to [0,1]",
        "composite": "compute_composite_score() with auto-renormalized default weights over available families",
    },
    "audit_files": ["calibration_item_audit.csv", "family_b_item_audit.csv"],
}

if len(FC_AUDIT) > 0:
    audit["axis_iii_t1_accuracy"] = round(float(FC_AUDIT["t1_correct"].mean()), 4)
    audit["axis_iii_t2_accuracy"] = round(float(FC_AUDIT["t2_correct"].mean()), 4)
    audit["audit_files"].append("family_c_item_audit.csv")

summary_path = os.path.join(output_dir, "benchmark_run_summary.json")
with open(summary_path, "w") as f:
    json.dump(audit, f, indent=2)

print(f"\nSummary: {summary_path}")
print(f"MetaScore: {metascore}")

## Scoring Methodology

### Axis I — Confidence Calibration (Monitoring)

Each item is scored using the **Brier scoring rule** (Brier, 1950):

$$\text{score} = 1 - (\text{confidence} - \text{outcome})^2$$

This is a **strictly proper scoring rule**: the expected score is uniquely maximised when stated confidence equals the model's true probability of being correct. A model cannot game this metric by hedging — it must be genuinely calibrated.

- **Correctness** is determined by a deterministic grading engine with 8 adjudication rules (exact match, numeric tolerance, alias expansion, etc.). No LLM judge is used.
- **Clean subset**: 12 items excluded where ≥3/5 models answered incorrectly (possible ambiguity in gold answer).

### Axis II — Selective Abstention (Control)

Each item is scored using a **utility payoff matrix** that maps (model action × gold action) → utility ∈ [−1, +1]:

| | Gold: answer | Gold: clarify | Gold: verify | Gold: abstain |
|---|---|---|---|---|
| **Answer (correct)** | +1.0 | +0.5 | +0.5 | −0.5 |
| **Answer (incorrect)** | −1.0 | −0.5 | −0.5 | −0.5 |
| **Clarify** | −0.2 | +1.0 | +0.3 | +0.3 |
| **Verify** | −0.2 | +0.3 | +1.0 | +0.3 |
| **Abstain** | −0.3 | +0.3 | +0.3 | +1.0 |

The headline metric is **UWAA** (Utility-Weighted Action Accuracy), normalised to [0, 1]:
$$\text{UWAA} = \frac{\text{mean utility} + 1}{2}$$

### Axis III — Self-Correction (Control)

Each item uses a **two-turn protocol**: the model answers (T1), then receives a review prompt and may revise (T2). Items are classified into transition types:

| Transition | Meaning |
|-----------|---------|
| **correction_gain** | Wrong → Right (genuine self-correction) |
| **maintain_correct** | Right → Right (stability) |
| **neutral_revision** | Wrong → Wrong with revision (no improvement) |
| **damage** | Right → Wrong (review caused error) |
| **stubborn_wrong** | Wrong → Wrong without revision (rigid) |
| **failed_revision** | Right → Right but unnecessary revision |

Base scores differ by subfamily: C1 (intrinsic, no evidence) penalizes damage more harshly (−0.40) than C2 (evidence-assisted, −0.25). Confidence adjustment in [−0.15, +0.10] rewards appropriate confidence changes. Raw scores are rescaled from [−0.65, 0.65] to [0, 1].

C1 and C2 are scored separately and contribute independently to the composite.

### Composite MetaScore

$$\text{MetaScore} = \text{compute\_composite\_score}(\text{calibration}, \text{UWAA}, \text{C1}, \text{C2})$$

Uses `compute_composite_score()` with default weights, auto-renormalized over available families. Effective weights with A+B+C: Calibration 40%, Abstention 27%, C1 (Intrinsic) 13%, C2 (Evidence) 20%. Monitoring-weighted (67% vs 33% control) per Nelson & Narens (1990). Weight optimization is deferred to a separate task.

### References

- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1–3.
- Burnell, R., et al. (2026). Measuring progress toward AGI. *DeepMind Technical Report*.
- Gneiting, T., & Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *JASA*, 102(477), 359–378.
- Huang, J., et al. (2024). Large language models cannot self-correct reasoning yet. *ICLR 2024*.
- Nelson, T. O., & Narens, L. (1990). Metamemory: A theoretical framework and new findings. *Psychology of Learning and Motivation*, 26, 125–173.

In [ ]:
%choose metajudge_metacognition_v1